# 05. Prediction: Static PE Malware Inference & Triage

## Security Policy: Zero File Execution
This inference workflow operates **strictly on static PE feature vectors**.
At no point is an executable file launched, spawned, or executed.
Predictions are made purely by evaluating structural and byte-level static features against our trained machine learning models.

### Workflow:
1. Load serialized model (`models/best_model.pkl`) and preprocessor (`models/preprocessor.pkl`).
2. Provide a 2381-dimensional EMBER static feature vector.
3. Compute predicted class (`BENIGN` or `MALICIOUS`) and probability confidence.
4. Integrate with `src.predict` reusable module.


In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import ember

# Add project root to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.predict import load_model, predict_from_vector, load_preprocessor

DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "ember2018")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

# 1. Load Best Model and Preprocessor
best_model = load_model("best_model.pkl", models_dir=MODELS_DIR)
preprocessor = load_preprocessor(models_dir=MODELS_DIR)
print("[+] Production model loaded:", type(best_model).__name__)


## Inference on Test Set Static Feature Vectors

In [ ]:
# Load sample feature vectors from test set
_, _, X_test, y_test = ember.read_vectorized_features(DATASET_DIR)

# Sample 5 benign and 5 malicious vectors
benign_indices = np.where(y_test == 0)[0][:5]
malicious_indices = np.where(y_test == 1)[0][:5]
sample_indices = np.concatenate([benign_indices, malicious_indices])

predictions = []
for idx in sample_indices:
    feat_vec = X_test[idx]
    true_label = int(y_test[idx])
    
    result = predict_from_vector(feat_vec, model=best_model)
    predictions.append({
        "Sample Index": idx,
        "True Label": "BENIGN" if true_label == 0 else "MALICIOUS",
        "Predicted": result["classification"],
        "Confidence": f"{result['confidence'] * 100:.2f}%",
        "Prob (Benign)": result["probabilities"]["benign"],
        "Prob (Malicious)": result["probabilities"]["malicious"],
        "Match": "✓ Correct" if (result["label"] == true_label) else "✗ Misclassified"
    })

df_preds = pd.DataFrame(predictions)
display(df_preds)


## Using `src.predict` as a Command-Line or Reusable Module
The `src/predict.py` module exposes clean helper functions for downstream integration, APIs, or SOC triage pipelines:
```python
from src.predict import predict_from_vector

result = predict_from_vector(feature_vector)
print(result['classification'])  # 'BENIGN' or 'MALICIOUS'
print(result['confidence'])      # e.g. 0.916
```


In [ ]:
# Test single vector prediction
test_sample = X_test[123]
res = predict_from_vector(test_sample)

print("Inference Result for Single Feature Vector:")
print(f"  Classification: {res['classification']}")
print(f"  Confidence:     {res['confidence'] * 100:.2f}%")
print(f"  Probabilities:  {res['probabilities']}")
